# 01 — Readout Calibration
Run after cooldown or if readout behaves unexpectedly.
Covers: TOF, resonator spectroscopy, chi to qubit, 2D gain sweep, single shot.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, 'C:\\Users\\Administrator\\Documents\\multimode_expts_tprocv2')

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from meas_utils import MultimodeStation
from calibration_helpers import (
    init_helpers,
    do_tof, update_tof,
    do_res_spec, update_res_spec,
    do_single_shot, update_single_shot,
)

station = MultimodeStation(
    experiment_name  = '2600403_lmm',
    hardware_config  = 'hardware_config_20260111.yml',
)
init_helpers(station)
config_thisrun = station.config_thisrun
cfg_dict       = station.cfg_dict
meas           = station.meas
print('Station ready.')

## Time of Flight

In [ ]:
tof = do_tof(
    config_thisrun                = config_thisrun,
    reps                          = 1,
    rounds                        = 100,
    check_e                       = False,
    final_delay                   = 500,
    analyze_and_display           = True,
    use_config_params_for_readout = False,
    gain                          = 1.0,
    length                        = 3.0,
    frequency                     = config_thisrun.device.readout.frequency,
)

In [ ]:
# Read x-value where signal rises from the plot, then set new_trig_offset
update_tof(
    tof             = tof,
    config_thisrun  = config_thisrun,
    new_trig_offset = 0.5,   # <-- edit this
)

## Resonator Spectroscopy

In [ ]:
rspec_wide = do_res_spec(
    config_thisrun                = config_thisrun,
    frequency                     = 7600.0,
    span                          = 30,
    expts                         = 300,
    reps                          = 100,
    rounds                        = 1,
    gain                          = 0.2,
    length                        = 3.0,
    final_delay                   = 250,
    pulse_e                       = False,
    prepulse                      = {},
    analyze_and_display           = True,
    use_config_params_for_readout = False,
)

In [ ]:
rspec_fine = do_res_spec(
    config_thisrun                = config_thisrun,
    frequency                     = 7600.391,   # <-- update from wide scan
    span                          = 3,
    expts                         = 300,
    reps                          = 400,
    rounds                        = 1,
    gain                          = 0.08,
    length                        = 3.0,
    final_delay                   = 250,
    pulse_e                       = False,
    prepulse                      = {},
    analyze_and_display           = True,
    use_config_params_for_readout = False,
)

In [ ]:
update_res_spec(
    rspec          = rspec_fine,
    config_thisrun = config_thisrun,
)

In [ ]:
# High power punchout check
rspec_highp = do_res_spec(
    config_thisrun                = config_thisrun,
    frequency                     = config_thisrun.device.readout.frequency,
    span                          = 5,
    expts                         = 300,
    reps                          = 100,
    rounds                        = 1,
    gain                          = 1.0,
    length                        = 3.0,
    final_delay                   = 250,
    pulse_e                       = False,
    prepulse                      = {},
    analyze_and_display           = True,
    use_config_params_for_readout = False,
)

## Chi to Qubit

In [ ]:
rspecs_chi = []
for pulse_e in [False, True]:
    rspec = do_res_spec(
        config_thisrun                = config_thisrun,
        frequency                     = config_thisrun.device.readout.frequency,
        span                          = 5,
        expts                         = 300,
        reps                          = 500,
        rounds                        = 1,
        gain                          = config_thisrun.device.readout.gain,
        length                        = config_thisrun.device.readout.length,
        final_delay                   = 250,
        pulse_e                       = pulse_e,
        prepulse                      = {},
        analyze_and_display           = False,
        use_config_params_for_readout = False,
    )
    rspecs_chi.append(rspec)

spec_analysis = meas.SpectroscopyFitting(data=rspecs_chi[0].data, station=station)
_ = spec_analysis.analyze(data_list=[x.data for x in rspecs_chi])
spec_analysis.display(
    data_list   = [x.data for x in rspecs_chi],
    data_labels = ['g state', 'e state'],
    title       = 'Chi to qubit',
    vlines      = [config_thisrun.device.readout.frequency],
    figsize     = (10, 8),
)
chi_ge = rspecs_chi[1].data['fit_amps'][2] - rspecs_chi[0].data['fit_amps'][2]
print(f'Chi_ge = {chi_ge:.3f} MHz')

## 2D Gain Sweep (Live Colorplot)

In [ ]:
gains      = np.logspace(-1, -4, 50)[::-1]
amp_matrix = None
freq_array = None
rspecs_2d  = []
peak_amps  = []

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
plt.ion()

for idx, gain in enumerate(gains):
    reps = min(int(100 * (1.0 / gain) ** 0.69897), 5000)
    print(f'[{idx+1}/{len(gains)}] gain={gain:.4f}, reps={reps}')

    rspec = do_res_spec(
        config_thisrun                = config_thisrun,
        frequency                     = config_thisrun.device.readout.frequency,
        span                          = 10,
        expts                         = 200,
        reps                          = reps,
        rounds                        = 1,
        gain                          = gain,
        length                        = config_thisrun.device.readout.length,
        final_delay                   = 250,
        pulse_e                       = False,
        prepulse                      = {},
        analyze_and_display           = False,
        use_config_params_for_readout = False,
    )
    rspecs_2d.append(rspec)

    xpts = rspec.data['xpts']
    amps = np.abs(np.array(rspec.data['avgi']) + 1j * np.array(rspec.data['avgq']))
    peak_amps.append(float(amps.max()))

    if amp_matrix is None:
        freq_array = xpts
        amp_matrix = np.full((len(gains), len(xpts)), np.nan)
    amp_matrix[idx] = amps

    axes[0].cla(); axes[1].cla()
    im = axes[0].pcolormesh(freq_array, gains[:idx+1], amp_matrix[:idx+1],
                            shading='auto', cmap='viridis')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Frequency (MHz)')
    axes[0].set_ylabel('Gain')
    axes[0].set_title(f'Res spec vs gain ({idx+1}/{len(gains)})')
    axes[1].loglog(gains[:idx+1], peak_amps, 'o-', color='C1')
    axes[1].set_xlabel('Gain')
    axes[1].set_ylabel('Peak amplitude')
    axes[1].grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    clear_output(wait=True)
    display(fig)

plt.ioff()
print('Gain sweep done.')

## Single Shot

In [ ]:
hst = do_single_shot(
    config_thisrun       = config_thisrun,
    readout_gain         = config_thisrun.device.readout.gain,
    readout_length       = config_thisrun.device.readout.length,
    readout_frequency    = config_thisrun.device.readout.frequency,
    shots                = 10000,
    pulse_e              = True,
    prepulse             = {},
    sweep_params         = {},
    final_delay          = 250,
    analyze_and_display  = True,
)

## Single Shot 2D Sweep (freq x gain)

In [ ]:
freq_span  = 3.0
gain_span  = 0.15
freq_expts = 30
gain_expts = 30

sweep_params_2d = {
    'readout_frequency': {
        'label':      'readout_pulse',
        'param':      'freq',
        'param_type': 'pulse',
        'expts':      freq_expts,
        'start':      config_thisrun.device.readout.frequency - freq_span / 2,
        'step':       freq_span / freq_expts,
    },
    'readout_gain': {
        'label':      'readout_pulse',
        'param':      'gain',
        'param_type': 'pulse',
        'expts':      gain_expts,
        'start':      0.05,
        'step':       gain_span / gain_expts,
    },
}

hst_sweep = do_single_shot(
    config_thisrun       = config_thisrun,
    readout_gain         = config_thisrun.device.readout.gain,
    readout_length       = config_thisrun.device.readout.length,
    readout_frequency    = config_thisrun.device.readout.frequency,
    shots                = 500,
    pulse_e              = True,
    prepulse             = {},
    sweep_params         = sweep_params_2d,
    final_delay          = 250,
    analyze_and_display  = False,
)

hst_analysis = meas.Histogram(
    hst_sweep.data,
    station      = station,
    sweep_params = sweep_params_2d,
)
_ = hst_analysis.analyze_swept_histogram_data()
plot_obj = hst_analysis.display_swept_histogram_data()
print('Optimal point:', plot_obj.results)

In [ ]:
# Uncomment to apply optimal readout point from sweep
# config_thisrun.device.readout.frequency = float(plot_obj.results[0]['max_y'])
# config_thisrun.device.readout.gain      = float(plot_obj.results[0]['max_x'])

In [ ]:
update_single_shot(
    hst_analysis   = hst,
    config_thisrun = config_thisrun,
)

In [ ]:
station.handle_config_update(write_to_file=True)
print('Config saved.')